# 🌬️ AirSpline — Sensores virtuales de PM₂.₅ con splines cúbicos
### Estudio de Caso 3 · *Air Health Analytics* · Análisis Numérico para Ingeniería

**Problema.** Un corredor vial de 9 km tiene 10 estaciones fijas (una cada 1 km). Se necesita estimar la concentración de PM₂.₅ cada **0.1 km (91 puntos)** para emitir alertas sanitarias geolocalizadas.

| Rol | Secciones del cuaderno |
|---|---|
| **Ingeniero de Modelado y Validación** | §2 Teoría (ecuaciones, fronteras, Runge) · §3 Spline propia vs SciPy · §5 LOOCV |
| **Ingeniero de Algoritmos y Simulación** | §4 Comparación Spline vs Hermite/PCHIP · §6 Malla fina, derivada, integral · §8 Monte Carlo |
| **Ingeniero de Integración y Estrategia** | §7 Política de alertas · §9 Diagrama de bloques · §10 KPIs · §11 Panel interactivo · §12 Pitch |

> ▶️ **Cómo ejecutar:** `Ejecutar todo` (VS Code: *Run All* · Colab: *Entorno de ejecución → Ejecutar todas*). La primera celda instala lo que falte. El **panel interactivo** está en la §11.

## 0. Configuración del entorno (VS Code / Colab)

In [ ]:
# Instala automáticamente las dependencias que falten (en Colab casi todo viene preinstalado)
import sys, subprocess, importlib

def _asegurar(paquete, modulo=None):
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        print(f"Instalando {paquete} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

for _p in ["numpy", "scipy", "matplotlib", "pandas", "ipywidgets", "jinja2"]:
    _asegurar(_p)

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    try:
        from google.colab import output
        output.enable_custom_widget_manager()   # habilita ipywidgets en Colab
    except Exception:
        pass
print("Entorno:", "Google Colab" if EN_COLAB else "Jupyter local / VS Code")

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from scipy.interpolate import (CubicSpline, CubicHermiteSpline, PchipInterpolator,
                               BarycentricInterpolator)
from scipy.integrate import simpson, trapezoid
import scipy, matplotlib

plt.rcParams.update({"figure.figsize": (11, 5), "figure.dpi": 100, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})
pd.set_option("display.precision", 3)

# Paleta e identidad visual del producto
C = {"dato": "#1f2937", "spline": "#2563eb", "natural": "#7c3aed", "hermite": "#ea580c",
     "pchip": "#059669", "lineal": "#6b7280", "lagrange": "#dc2626",
     "amarilla": "#facc15", "roja": "#ef4444", "banda": "#93c5fd"}
print(f"numpy {np.__version__} · scipy {scipy.__version__} · matplotlib {matplotlib.__version__}")

## 1. Datos del corredor

In [ ]:
# 10 estaciones equidistantes (h = 1 km) — concentración PM2.5 en µg/m³
x = np.arange(0.0, 10.0, 1.0)                                    # km 0 … 9
y = np.array([45, 52, 68, 85, 92, 88, 79, 63, 48, 42], dtype=float)
estaciones = [f"E{i+1}" for i in range(len(x))]

L = x[-1] - x[0]                                  # longitud del corredor = 9 km
DX = 0.1                                          # paso de la malla fina
xf = np.round(np.linspace(x[0], x[-1], int(round(L / DX)) + 1), 1)   # 91 puntos

UMBRAL_AMARILLA, UMBRAL_ROJA = 70.0, 90.0         # µg/m³
SIGMA, N_MC, SEMILLA = 2.5, 100, 2024             # Monte Carlo

datos = pd.DataFrame({"Estación": estaciones, "Distancia (km)": x, "PM2.5 (µg/m³)": y})
print(f"Malla fina: {len(xf)} puntos, Δx = {DX} km  →  {len(xf) - len(x)} sensores virtuales")
datos.set_index("Estación").T

## 2. Marco teórico (Ingeniero de Modelado y Validación)

### 2.1 Spline cúbico
En cada tramo $[x_i, x_{i+1}]$, $i=0,\dots,n-1$ (aquí $n=9$, $h=1$ km):

$$S_i(x)=a_i+b_i(x-x_i)+c_i(x-x_i)^2+d_i(x-x_i)^3$$

Condiciones (4n incógnitas): interpolación $S_i(x_i)=y_i$, $S_i(x_{i+1})=y_{i+1}$; continuidad de $S'$ y $S''$ en los nodos internos (**clase C²**) y **2 condiciones de frontera**. Con $a_i=y_i$ y $c_i=S''(x_i)/2$, el problema se reduce al sistema **tridiagonal, estrictamente diagonal dominante** (solución única, algoritmo de Thomas $O(n)$):

$$h_{i-1}c_{i-1}+2(h_{i-1}+h_i)c_i+h_i c_{i+1}=\frac{3}{h_i}(a_{i+1}-a_i)-\frac{3}{h_{i-1}}(a_i-a_{i-1})$$

$$b_i=\frac{a_{i+1}-a_i}{h_i}-\frac{h_i}{3}(2c_i+c_{i+1}),\qquad d_i=\frac{c_{i+1}-c_i}{3h_i}$$

### 2.2 Condiciones de frontera evaluadas
| Tipo | Condición | Comentario para el corredor |
|---|---|---|
| **Natural** | $S''(0)=S''(9)=0$ | Supone curvatura nula en los extremos (sin información fuera del corredor). Sesga los tramos de borde. |
| **Not-a-knot** | $S'''$ continua en $x_1$ y $x_{n-1}$ | No impone nada físico; deja que los datos determinen la curvatura de borde. |
| **Sujeta (clamped)** | $S'(0)=f'_0,\ S'(9)=f'_9$ | Requiere pendientes; aquí se estiman con `np.gradient` (diferencias finitas). |
| **Periódica** | $S(0)=S(9)$, $S',S''$ iguales | **No aplica**: el corredor es abierto y $y_0=45\neq y_9=42$. |

La elección final se hace **con evidencia**: la condición con menor RMSE en la validación cruzada (§5).

### 2.3 ¿Por qué no un polinomio de grado alto? — Fenómeno de Runge
El interpolante de Lagrange de grado 9 tiene error

$$f(x)-P_9(x)=\frac{f^{(10)}(\xi)}{10!}\prod_{i=0}^{9}(x-x_i)$$

Con nodos **equidistantes**, el factor $\prod(x-x_i)$ crece fuertemente cerca de los extremos y la constante de Lebesgue crece como $\Lambda_n\sim \dfrac{2^{n+1}}{e\,n\ln n}$: pequeñas perturbaciones (ruido del sensor) se amplifican y aparecen **oscilaciones espurias** en los bordes. El spline cúbico evita esto porque:
1. Usa polinomios de **grado 3 por tramos** (grado fijo, no crece con el número de estaciones).
2. Es **local**: el efecto de un dato decae rápidamente con la distancia.
3. Tiene la propiedad de **curvatura mínima**: entre todas las funciones C² que interpolan los datos, el spline natural minimiza $\int (f'')^2\,dx$.
4. Converge con error $O(h^4)$ al refinar la red de estaciones.

## 3. Implementación propia del spline natural (algoritmo de Thomas) y verificación con SciPy

In [ ]:
def spline_natural_coeficientes(x, y):
    # Coeficientes (a, b, c, d) de cada tramo del spline cúbico natural.
    # Resuelve el sistema tridiagonal con el algoritmo de Thomas.
    n = len(x) - 1
    h = np.diff(x)
    a = y.astype(float).copy()
    # Lado derecho (sólo nodos internos)
    alpha = np.zeros(n + 1)
    for i in range(1, n):
        alpha[i] = 3 / h[i] * (a[i + 1] - a[i]) - 3 / h[i - 1] * (a[i] - a[i - 1])
    # Eliminación hacia adelante (Thomas) con c0 = cn = 0 (frontera natural)
    l, mu, z = np.ones(n + 1), np.zeros(n + 1), np.zeros(n + 1)
    for i in range(1, n):
        l[i] = 2 * (x[i + 1] - x[i - 1]) - h[i - 1] * mu[i - 1]
        mu[i] = h[i] / l[i]
        z[i] = (alpha[i] - h[i - 1] * z[i - 1]) / l[i]
    # Sustitución hacia atrás
    c, b, d = np.zeros(n + 1), np.zeros(n), np.zeros(n)
    for j in range(n - 1, -1, -1):
        c[j] = z[j] - mu[j] * c[j + 1]
        b[j] = (a[j + 1] - a[j]) / h[j] - h[j] * (c[j + 1] + 2 * c[j]) / 3
        d[j] = (c[j + 1] - c[j]) / (3 * h[j])
    return a[:-1], b, c[:-1], d

a_, b_, c_, d_ = spline_natural_coeficientes(x, y)
tabla_coef = pd.DataFrame({"Tramo [km]": [f"[{x[i]:.0f}, {x[i+1]:.0f}]" for i in range(len(x) - 1)],
                           "a_i": a_, "b_i": b_, "c_i": c_, "d_i": d_})

# Verificación contra scipy (CubicSpline guarda coeficientes en orden d, c, b, a)
cs_nat_ref = CubicSpline(x, y, bc_type="natural")
dif = np.max(np.abs(cs_nat_ref.c[::-1].T - np.column_stack([a_, b_, c_, d_])))
print(f"Diferencia máxima entre implementación propia y scipy: {dif:.2e}  ✔")
tabla_coef

## 4. Modelos de interpolación y comparación gráfica (Ingeniero de Algoritmos)

In [ ]:
def construir_modelos(x, y):
    # Devuelve un diccionario {nombre: interpolante} entrenado con (x, y).
    pend = np.gradient(y, x)          # pendientes espaciales por diferencias finitas (admite malla no uniforme)
    return {
        "Spline not-a-knot":   CubicSpline(x, y, bc_type="not-a-knot"),
        "Spline natural":      CubicSpline(x, y, bc_type="natural"),
        "Spline sujeta":       CubicSpline(x, y, bc_type=((1, pend[0]), (1, pend[-1]))),
        "Hermite (np.gradient)": CubicHermiteSpline(x, y, pend),
        "PCHIP":               PchipInterpolator(x, y),
        "Lineal":              (lambda t, x=x, y=y: np.interp(t, x, y)),
        "Lagrange (grado n-1)": BarycentricInterpolator(x, y),
    }

modelos = construir_modelos(x, y)
ESTILO = {"Spline not-a-knot": (C["spline"], "-", 2.6), "Spline natural": (C["natural"], "--", 1.8),
          "Spline sujeta": ("#0891b2", ":", 1.8), "Hermite (np.gradient)": (C["hermite"], "-.", 1.8),
          "PCHIP": (C["pchip"], "-", 1.4), "Lineal": (C["lineal"], "-", 1.0),
          "Lagrange (grado n-1)": (C["lagrange"], ":", 1.6)}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1.6, 1]})
for nombre, f in modelos.items():
    col, ls, lw = ESTILO[nombre]
    ax1.plot(xf, f(xf), ls, color=col, lw=lw, label=nombre)
ax1.plot(x, y, "o", color=C["dato"], ms=8, zorder=5, label="Estaciones")
for xi, yi, e in zip(x, y, estaciones):
    ax1.annotate(e, (xi, yi), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8)
ax1.axhline(UMBRAL_AMARILLA, color=C["amarilla"], lw=1); ax1.axhline(UMBRAL_ROJA, color=C["roja"], lw=1)
ax1.set(title="PM₂.₅ a lo largo del corredor — comparación de interpolantes",
        xlabel="Distancia (km)", ylabel="PM₂.₅ (µg/m³)")
ax1.legend(fontsize=8, ncol=2)

# Continuidad C²: la segunda derivada del spline es continua; la de Hermite/PCHIP salta en los nodos
for nombre in ["Spline not-a-knot", "Hermite (np.gradient)", "PCHIP"]:
    col, ls, lw = ESTILO[nombre]
    xx = np.linspace(0, 9, 2000)
    ax2.plot(xx, modelos[nombre].derivative(2)(xx), color=col, lw=1.6, label=nombre)
for xi in x: ax2.axvline(xi, color="k", lw=0.4, alpha=0.3)
ax2.set(title="Segunda derivada S''(x): C² (spline) vs C¹ (Hermite)", xlabel="Distancia (km)",
        ylabel="S'' (µg/m³/km²)")
ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

# Diferencia puntual Spline vs Hermite
dif_sh = np.abs(modelos["Spline not-a-knot"](xf) - modelos["Hermite (np.gradient)"](xf))
print(f"Máxima diferencia Spline vs Hermite en la malla fina: {dif_sh.max():.2f} µg/m³ en km {xf[dif_sh.argmax()]:.1f}")

In [ ]:
# Demostración clásica del fenómeno de Runge: f(x) = 1/(1+25x²), 10 nodos equidistantes
f_runge = lambda t: 1 / (1 + 25 * t**2)
xr = np.linspace(-1, 1, 10); xx = np.linspace(-1, 1, 600)
P = BarycentricInterpolator(xr, f_runge(xr)); S = CubicSpline(xr, f_runge(xr))

# Sensibilidad al ruido de los sensores en el problema real (σ = 2.5 µg/m³)
rng_demo = np.random.default_rng(7)
y_ruido = y + rng_demo.normal(0, SIGMA, len(y))
xext = np.linspace(-0.3, 9.3, 600)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 4.6))
a1.plot(xx, f_runge(xx), "k", lw=2, label="f(x) real")
a1.plot(xx, P(xx), color=C["lagrange"], label="Lagrange grado 9")
a1.plot(xx, S(xx), color=C["spline"], ls="--", label="Spline cúbico")
a1.plot(xr, f_runge(xr), "ko")
a1.set(title="Fenómeno de Runge (función de prueba)", xlabel="x"); a1.legend(fontsize=9)
a2.plot(xext, BarycentricInterpolator(x, y_ruido)(xext), color=C["lagrange"], label="Lagrange (datos con ruido)")
a2.plot(xext, CubicSpline(x, y_ruido)(xext), color=C["spline"], label="Spline (datos con ruido)")
a2.plot(x, y_ruido, "ko", label="Lecturas con ruido σ=2.5")
a2.set(title="Corredor: sensibilidad al ruido cerca de los bordes", xlabel="km", ylabel="µg/m³"); a2.legend(fontsize=9)
plt.tight_layout(); plt.show()
print(f"Error máx. Runge — Lagrange: {np.max(np.abs(P(xx)-f_runge(xx))):.3f} | Spline: {np.max(np.abs(S(xx)-f_runge(xx))):.3f}")

## 5. Validación cruzada *Leave-One-Out* en los 8 nodos internos (Ingeniero de Modelado)
Para cada estación interna $E_2,\dots,E_9$: se retira, se ajusta el modelo con las 9 restantes y se predice la estación retirada.

$$e_i=|\hat y_{-i}(x_i)-y_i|,\qquad \text{RMSE}=\sqrt{\tfrac{1}{8}\sum e_i^2},\qquad e_{\max}=\max_i e_i$$

In [ ]:
def loocv(x, y, nombres=None):
    # Leave-One-Out en los nodos internos. Devuelve (tabla de errores absolutos, resumen de métricas).
    nombres = nombres or list(construir_modelos(x, y).keys())
    filas = []
    for i in range(1, len(x) - 1):                       # sólo nodos internos (no se extrapola)
        m = np.ones(len(x), dtype=bool); m[i] = False
        mods = construir_modelos(x[m], y[m])
        fila = {"Estación": f"E{i+1}", "km": x[i], "Real": y[i]}
        for n in nombres:
            pred = float(mods[n](x[i]))
            fila[n] = abs(pred - y[i])
        filas.append(fila)
    errores = pd.DataFrame(filas).set_index("Estación")
    e = errores[nombres]
    resumen = pd.DataFrame({"RMSE": np.sqrt((e**2).mean()), "MAE": e.mean(),
                            "Error máx.": e.max(), "Estación peor": e.idxmax()}).sort_values("RMSE")
    return errores, resumen

errores_cv, resumen_cv = loocv(x, y)
print("Errores absolutos de predicción LOOCV (µg/m³):")
display(errores_cv.style.format(precision=2).background_gradient(cmap="Reds",
        subset=list(resumen_cv.index)))
print("Resumen de métricas (ordenado por RMSE):")
display(resumen_cv.style.format({"RMSE": "{:.3f}", "MAE": "{:.3f}", "Error máx.": "{:.3f}"})
        .highlight_min(subset=["RMSE", "MAE", "Error máx."], color="#bbf7d0"))

In [ ]:
# Selección de la condición de frontera con evidencia numérica
splines_cv = resumen_cv.loc[[n for n in resumen_cv.index if n.startswith("Spline")]]
MODELO_PRINCIPAL = splines_cv["RMSE"].idxmin()
BC_PRINCIPAL = {"Spline not-a-knot": "not-a-knot", "Spline natural": "natural",
                "Spline sujeta": "clamped"}[MODELO_PRINCIPAL]
rmse_lin = resumen_cv.loc["Lineal", "RMSE"]; rmse_best = resumen_cv.loc[MODELO_PRINCIPAL, "RMSE"]
rmse_lag = resumen_cv.loc["Lagrange (grado n-1)", "RMSE"]
REDUCCION_RMSE = 100 * (1 - rmse_best / rmse_lin)
print(f"Modelo principal elegido: {MODELO_PRINCIPAL}  (RMSE LOOCV = {rmse_best:.3f} µg/m³)")
print(f"Reducción de RMSE frente a interpolación lineal: {REDUCCION_RMSE:.1f} %")
print(f"Reducción de RMSE frente a Lagrange grado 8:   {100*(1-rmse_best/rmse_lag):.1f} %")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 4.4))
orden = resumen_cv.index
a1.barh(orden, resumen_cv["RMSE"], color=[ESTILO[n][0] for n in orden])
a1.invert_yaxis(); a1.set(title="RMSE LOOCV por método (menor es mejor)", xlabel="µg/m³")
for n in ["Spline not-a-knot", "Spline natural", "Hermite (np.gradient)", "Lineal", "Lagrange (grado n-1)"]:
    a2.plot(errores_cv["km"], errores_cv[n], "o-", color=ESTILO[n][0], label=n)
a2.set(title="Error absoluto por estación retirada", xlabel="km", ylabel="µg/m³"); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Interpretación.** La frontera *natural* fuerza $S''=0$ en los km 0 y 9, pero el perfil de contaminación sí tiene curvatura allí; el error se concentra en las estaciones vecinas a los bordes (E2, E9). La condición *not-a-knot* no impone esa hipótesis artificial y suele obtener el menor RMSE. Lagrange se degrada al perder un nodo (grado 8 con nodos no equidistantes), y la interpolación lineal pierde los picos porque no captura curvatura.

## 6. Malla fina (91 puntos), derivada espacial, integral y concentración media (Ingeniero de Algoritmos)

In [ ]:
cs = CubicSpline(x, y, bc_type=BC_PRINCIPAL if BC_PRINCIPAL != "clamped"
                 else ((1, np.gradient(y, x)[0]), (1, np.gradient(y, x)[-1])))
herm = modelos["Hermite (np.gradient)"]
dcs = cs.derivative()

def nivel_alerta(c, ua=UMBRAL_AMARILLA, ur=UMBRAL_ROJA):
    return np.where(c >= ur, "ROJA", np.where(c >= ua, "AMARILLA", "Normal"))

malla = pd.DataFrame({"km": xf, "Spline (µg/m³)": cs(xf), "Hermite (µg/m³)": herm(xf),
                      "dC/dx (µg/m³/km)": dcs(xf)})
malla["Alerta"] = nivel_alerta(malla["Spline (µg/m³)"].values)
malla["Tipo"] = np.where(np.isin(xf, x), "Estación física", "Sensor virtual")

# Integral definida: carga total acumulada y concentración media del corredor
carga_total = cs.integrate(x[0], x[-1])            # µg·km/m³
C_media = carga_total / L
carga_herm = herm.integrate(x[0], x[-1])
carga_simpson = simpson(cs(xf), x=xf)
carga_trap_datos = trapezoid(y, x)

# Máximo de la curva: raíz de S'(x) = 0
crit = dcs.roots(extrapolate=False)
crit = crit[(crit >= x[0]) & (crit <= x[-1])]
x_max = crit[np.argmax(cs(crit))] if len(crit) else xf[np.argmax(cs(xf))]
i_grad = np.argmax(np.abs(dcs(xf)))

resultados = pd.DataFrame({
    "Magnitud": ["Carga total ∫₀⁹ S(x)dx (spline, exacta)", "Carga total (Hermite)",
                 "Carga total (Simpson sobre malla 0.1 km)", "Carga total (trapecio sobre 10 estaciones)",
                 "Concentración media del corredor C̄ = ∫S/L", "Pico máximo de PM2.5 (S'(x)=0)",
                 "Gradiente espacial máximo |dC/dx|"],
    "Valor": [carga_total, carga_herm, carga_simpson, carga_trap_datos, C_media, float(cs(x_max)),
              abs(dcs(xf[i_grad]))],
    "Unidad / ubicación": ["µg·km/m³", "µg·km/m³", "µg·km/m³", "µg·km/m³", "µg/m³",
                           f"µg/m³ en km {x_max:.3f}", f"µg/m³/km en km {xf[i_grad]:.1f}"]})
display(resultados.style.format({"Valor": "{:.3f}"}).hide(axis="index"))

malla.to_csv("airspline_malla_91_puntos.csv", index=False)
print("Malla fina exportada a airspline_malla_91_puntos.csv. Primeras filas:")
display(malla.head(12).style.format(precision=2).hide(axis="index"))

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 7.5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
a1.fill_between(xf, cs(xf), color=C["banda"], alpha=0.35, label=f"Carga total = {carga_total:.1f} µg·km/m³")
a1.plot(xf, cs(xf), color=C["spline"], lw=2.5, label=MODELO_PRINCIPAL)
a1.plot(xf[~np.isin(xf, x)], cs(xf[~np.isin(xf, x)]), ".", color=C["spline"], ms=3, label="81 sensores virtuales")
a1.plot(x, y, "o", color=C["dato"], ms=8, label="10 estaciones físicas")
a1.axhline(C_media, color="k", ls="--", lw=1.2, label=f"C̄ = {C_media:.2f} µg/m³")
a1.plot(x_max, cs(x_max), "*", color=C["roja"], ms=16, label=f"Pico {float(cs(x_max)):.1f} µg/m³ (km {x_max:.2f})")
a1.set(ylabel="PM₂.₅ (µg/m³)", title="Perfil interpolado, carga acumulada y concentración media"); a1.legend(fontsize=9)
a2.plot(xf, dcs(xf), color=C["hermite"], lw=2, label="dC/dx spline (analítica)")
a2.plot(x, np.gradient(y, x), "s", color="k", ms=5, label="np.gradient en estaciones")
a2.axhline(0, color="k", lw=0.8)
a2.set(xlabel="Distancia (km)", ylabel="µg/m³/km", title="Gradiente espacial de contaminación"); a2.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 7. Política de gestión de riesgo ambiental (Ingeniero de Integración y Estrategia)
| Nivel | Criterio | Acción sugerida |
|---|---|---|
| 🟢 Normal | $C < 70$ µg/m³ | Información general |
| 🟡 **Alerta Amarilla** | $70 \le C < 90$ µg/m³ | Aviso a grupos sensibles, ciclistas: rutas alternas |
| 🔴 **Alerta Roja** | $C \ge 90$ µg/m³ | Suspender actividad física al aire libre en escuelas, aviso a hospitales |

Los límites exactos de cada tramo se obtienen resolviendo $S(x)=U$ sobre los polinomios por tramos (`CubicSpline.solve`), no sólo sobre la malla de 0.1 km.

In [ ]:
def tramos_sobre_umbral(f, umbral, a=0.0, b=9.0):
    # Intervalos [x_ini, x_fin] donde f(x) >= umbral. Raíces exactas si f es un polinomio por tramos.
    if hasattr(f, "solve"):
        r = np.asarray(f.solve(umbral, extrapolate=False), dtype=float)
    else:                                                   # interpolante genérico: bisección en malla densa
        xx = np.linspace(a, b, 9001); g = f(xx) - umbral
        idx = np.where(np.sign(g[:-1]) * np.sign(g[1:]) < 0)[0]
        r = xx[idx] - g[idx] * (xx[idx + 1] - xx[idx]) / (g[idx + 1] - g[idx])
    r = r[np.isfinite(r) & (r > a) & (r < b)]
    pts = np.unique(np.concatenate([[a], r, [b]]))
    return [(p, q) for p, q in zip(pts[:-1], pts[1:]) if f(0.5 * (p + q)) >= umbral]

def tabla_alertas(f, ua=UMBRAL_AMARILLA, ur=UMBRAL_ROJA, a=0.0, b=9.0):
    filas = []
    rojas = tramos_sobre_umbral(f, ur, a, b)
    for p, q in tramos_sobre_umbral(f, ua, a, b):
        filas.append(("🟡 Amarilla o superior (≥ %.0f)" % ua, p, q))
    for p, q in rojas:
        filas.append(("🔴 Roja (≥ %.0f)" % ur, p, q))
    # sub-tramos exclusivamente amarillos = amarilla − roja
    for p, q in tramos_sobre_umbral(f, ua, a, b):
        cortes = [p] + [v for r in rojas for v in r if p < v < q] + [q]
        for s, t in zip(cortes[:-1], cortes[1:]):
            if f(0.5 * (s + t)) < ur:
                filas.append(("🟡 Sólo amarilla [%.0f, %.0f)" % (ua, ur), s, t))
    df = pd.DataFrame(filas, columns=["Nivel", "Desde km", "Hasta km"])
    df["Longitud (km)"] = df["Hasta km"] - df["Desde km"]
    df["% corredor"] = 100 * df["Longitud (km)"] / (b - a)
    return df

alertas = tabla_alertas(cs)
display(alertas.style.format({"Desde km": "{:.3f}", "Hasta km": "{:.3f}", "Longitud (km)": "{:.3f}",
                              "% corredor": "{:.1f}"}).hide(axis="index"))

# Alertas operativas sobre la malla de 0.1 km (lo que recibe la app de los usuarios)
for nivel in ["AMARILLA", "ROJA"]:
    km = malla.loc[malla["Alerta"] == nivel, "km"].values
    print(f"Puntos de malla en alerta {nivel:8s}: {len(km):2d}  →  km {km.min():.1f} – {km.max():.1f}" if len(km)
          else f"Sin puntos en alerta {nivel}")

In [ ]:
def dibujar_mapa_alertas(ax, f, xf, ua=UMBRAL_AMARILLA, ur=UMBRAL_ROJA, titulo=""):
    c = f(xf)
    ax.plot(xf, c, color=C["spline"], lw=2.5, label="PM₂.₅ interpolado")
    ax.fill_between(xf, ua, c, where=c >= ua, color=C["amarilla"], alpha=0.45, interpolate=True, label="Alerta amarilla")
    ax.fill_between(xf, ur, c, where=c >= ur, color=C["roja"], alpha=0.6, interpolate=True, label="Alerta roja")
    ax.axhline(ua, color="#ca8a04", ls="--", lw=1); ax.axhline(ur, color="#b91c1c", ls="--", lw=1)
    for p, q in tramos_sobre_umbral(f, ur, xf[0], xf[-1]):
        ax.axvspan(p, q, color=C["roja"], alpha=0.08)
        ax.annotate(f"ROJA km {p:.2f}–{q:.2f}", ((p + q) / 2, ur - 7), ha="center", color="#b91c1c", fontsize=9, weight="bold")
    for p, q in tramos_sobre_umbral(f, ua, xf[0], xf[-1]):
        ax.annotate(f"AMARILLA km {p:.2f}–{q:.2f}", ((p + q) / 2, ua - 7), ha="center", color="#a16207", fontsize=9, weight="bold")
    ax.set(xlabel="Distancia en el corredor (km)", ylabel="PM₂.₅ (µg/m³)", title=titulo)

fig, (ax, axb) = plt.subplots(2, 1, figsize=(13, 6.3), gridspec_kw={"height_ratios": [4, 0.6]}, sharex=True)
dibujar_mapa_alertas(ax, cs, xf, titulo="Mapa de alertas sanitarias del corredor (AirSpline)")
ax.plot(x, y, "o", color=C["dato"], ms=7, label="Estaciones"); ax.legend(fontsize=9, loc="upper right")
# Franja tipo "semáforo" sobre la avenida, celda = 0.1 km
colores = {"Normal": "#22c55e", "AMARILLA": C["amarilla"], "ROJA": C["roja"]}
axb.bar(xf, 1, width=DX, color=[colores[a] for a in malla["Alerta"]], edgecolor="none")
axb.set_yticks([]); axb.set_ylabel("Avenida", rotation=0, ha="right"); axb.grid(False)
plt.tight_layout(); plt.show()

## 8. Mejora tecnológica: banda de incertidumbre por Monte Carlo / *bootstrap* paramétrico
Modelo de ruido instrumental: $\tilde y_i^{(k)} = y_i + \varepsilon_i^{(k)}$, $\varepsilon \sim N(0,\sigma^2)$, $\sigma=2.5$ µg/m³, $k=1,\dots,100$.
En cada realización se ajusta un spline cúbico y se evalúa en la malla fina. La banda del 95 % se forma con los percentiles 2.5 y 97.5 en cada punto; además se estima la **probabilidad de excedencia** $P(C(x)\ge 90)$.

**Decisión robusta:** 🔴 *Roja confirmada* si el límite inferior de la banda ≥ 90 · 🔴 *Roja probable* si $P(C\ge 90)\ge 50\%$ · 🟠 *Roja precautoria* si sólo el límite superior ≥ 90. Con un pico nominal de ≈92 µg/m³ y ruido σ = 2.5, la alerta roja **no** se puede confirmar al 95 %: la recomendación robusta es activar la roja precautoria en todo el tramo donde $P97.5 \ge 90$.

In [ ]:
def monte_carlo(x, y, xf, sigma=SIGMA, n=N_MC, semilla=SEMILLA, bc=BC_PRINCIPAL, ur=UMBRAL_ROJA, ua=UMBRAL_AMARILLA):
    rng = np.random.default_rng(semilla)
    Y = y + rng.normal(0.0, sigma, size=(n, len(y)))          # n realizaciones perturbadas
    curvas, cargas, rojas_ini, rojas_fin = [], [], [], []
    for yk in Y:
        bck = bc if bc != "clamped" else ((1, np.gradient(yk, x)[0]), (1, np.gradient(yk, x)[-1]))
        sk = CubicSpline(x, yk, bc_type=bck)
        curvas.append(sk(xf)); cargas.append(sk.integrate(x[0], x[-1]))
        tr = tramos_sobre_umbral(sk, ur, x[0], x[-1])
        if tr:
            rojas_ini.append(tr[0][0]); rojas_fin.append(tr[-1][1])
    curvas = np.array(curvas)
    return {"Y": Y, "curvas": curvas, "media": curvas.mean(0),
            "lo": np.percentile(curvas, 2.5, axis=0), "hi": np.percentile(curvas, 97.5, axis=0),
            "p_roja": (curvas >= ur).mean(0), "p_amarilla": (curvas >= ua).mean(0),
            "cargas": np.array(cargas), "rojas_ini": np.array(rojas_ini), "rojas_fin": np.array(rojas_fin),
            "frac_con_roja": len(rojas_ini) / n}

mc = monte_carlo(x, y, xf)
ancho = mc["hi"] - mc["lo"]
roja_conf = xf[mc["lo"] >= UMBRAL_ROJA]; roja_prec = xf[mc["hi"] >= UMBRAL_ROJA]
print(f"Realizaciones: {N_MC} · σ = {SIGMA} µg/m³ · semilla = {SEMILLA}")
print(f"Ancho medio de la banda 95 %: {ancho.mean():.2f} µg/m³ (máx {ancho.max():.2f} en km {xf[ancho.argmax()]:.1f}, mín {ancho.min():.2f} en km {xf[ancho.argmin()]:.1f})")
print(f"Realizaciones que activan Alerta Roja: {100*mc['frac_con_roja']:.0f} %")
if len(mc["rojas_ini"]):
    print(f"Inicio tramo rojo: {mc['rojas_ini'].mean():.3f} ± {mc['rojas_ini'].std(ddof=1):.3f} km · "
          f"Fin: {mc['rojas_fin'].mean():.3f} ± {mc['rojas_fin'].std(ddof=1):.3f} km")
print("Roja CONFIRMADA (P2.5 ≥ 90):  " + (f"km {roja_conf.min():.1f} – {roja_conf.max():.1f}" if len(roja_conf) else "ningún punto"))
roja_prob = xf[mc["p_roja"] >= 0.5]
print("Roja PROBABLE (P(C≥90) ≥ 50 %): " + (f"km {roja_prob.min():.1f} – {roja_prob.max():.1f}" if len(roja_prob) else "ningún punto"))
print("Roja PRECAUTORIA (P97.5 ≥ 90): " + (f"km {roja_prec.min():.1f} – {roja_prec.max():.1f}" if len(roja_prec) else "ningún punto"))
IC_carga = np.percentile(mc["cargas"], [2.5, 97.5])
print(f"Concentración media del corredor: {C_media:.2f} µg/m³, IC95 % [{IC_carga[0]/L:.2f}, {IC_carga[1]/L:.2f}]")

In [ ]:
fig = plt.figure(figsize=(15, 9))
gs = fig.add_gridspec(2, 3, height_ratios=[1.6, 1])
ax = fig.add_subplot(gs[0, :])
for k in range(min(40, N_MC)):
    ax.plot(xf, mc["curvas"][k], color="#64748b", lw=0.5, alpha=0.25)
ax.fill_between(xf, mc["lo"], mc["hi"], color=C["banda"], alpha=0.55, label="Banda de confianza 95 %")
ax.plot(xf, cs(xf), color=C["spline"], lw=2.5, label="Spline (lecturas nominales)")
ax.plot(x, y, "o", color=C["dato"], ms=7, label="Estaciones")
ax.axhline(UMBRAL_AMARILLA, color="#ca8a04", ls="--", lw=1, label="70 µg/m³")
ax.axhline(UMBRAL_ROJA, color="#b91c1c", ls="--", lw=1, label="90 µg/m³")
if len(roja_prec): ax.axvspan(roja_prec.min(), roja_prec.max(), color="orange", alpha=0.15, label="Roja precautoria")
if len(roja_conf): ax.axvspan(roja_conf.min(), roja_conf.max(), color=C["roja"], alpha=0.2, label="Roja confirmada")
ax.set(title=f"Monte Carlo ({N_MC} realizaciones, σ = {SIGMA} µg/m³): banda del 95 % y zonas de decisión",
       xlabel="km", ylabel="PM₂.₅ (µg/m³)"); ax.legend(fontsize=8, ncol=4, loc="lower center")

a1 = fig.add_subplot(gs[1, 0])
a1.plot(xf, 100 * mc["p_roja"], color=C["roja"], lw=2, label="P(C ≥ 90)")
a1.plot(xf, 100 * mc["p_amarilla"], color="#ca8a04", lw=2, label="P(C ≥ 70)")
a1.set(title="Probabilidad de excedencia", xlabel="km", ylabel="%"); a1.legend(fontsize=8)
a2 = fig.add_subplot(gs[1, 1])
a2.plot(xf, ancho, color=C["spline"]); a2.plot(x, ancho[np.isin(xf, x)], "ko", ms=4)
a2.set(title="Ancho de la banda 95 % (µg/m³)", xlabel="km")
a3 = fig.add_subplot(gs[1, 2])
if len(mc["rojas_ini"]):
    a3.hist(mc["rojas_ini"], bins=15, color=C["roja"], alpha=0.7, label="inicio tramo rojo")
    a3.hist(mc["rojas_fin"], bins=15, color="#7f1d1d", alpha=0.7, label="fin tramo rojo")
a3.set(title="Incertidumbre en los límites de la Alerta Roja", xlabel="km", ylabel="frecuencia"); a3.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 9. Arquitectura de la mejora — diagrama de bloques (Ingeniero de Integración)

In [ ]:
def diagrama_bloques():
    fig, ax = plt.subplots(figsize=(15, 4.6)); ax.set_xlim(0, 15); ax.set_ylim(0, 4.6); ax.axis("off")
    bloques = [
        (0.2, "10 estaciones\nPM₂.₅ (1 km)", "#e0e7ff"),
        (2.4, "Ingesta + control\nde calidad (QC)", "#e0f2fe"),
        (4.6, "Motor AirSpline\nCubicSpline C²\n(91 puntos)", "#dbeafe"),
        (6.8, "Monte Carlo\nσ=2.5, N=100\nbanda 95 %", "#fef3c7"),
        (9.0, "Motor de alertas\n70 / 90 µg/m³\nP(excedencia)", "#fee2e2"),
        (11.2, "API + Dashboard\n+ notificaciones", "#dcfce7"),
    ]
    w, h, y0 = 1.9, 1.5, 2.4
    for i, (x0, txt, col) in enumerate(bloques):
        ax.add_patch(FancyBboxPatch((x0, y0), w, h, boxstyle="round,pad=0.05", fc=col, ec="#334155", lw=1.4))
        ax.text(x0 + w / 2, y0 + h / 2, txt, ha="center", va="center", fontsize=9.5, weight="bold")
        if i < len(bloques) - 1:
            ax.annotate("", (bloques[i + 1][0] - 0.05, y0 + h / 2), (x0 + w + 0.05, y0 + h / 2),
                        arrowprops=dict(arrowstyle="-|>", lw=1.6, color="#334155"))
    usuarios = [("🏫 Escuelas", 10.2), ("🏥 Hospitales", 12.0), ("🚴 Ciclistas", 13.8)]
    for txt, xu in usuarios:
        ax.add_patch(FancyBboxPatch((xu - 0.8, 0.3), 1.6, 0.8, boxstyle="round,pad=0.05", fc="#f1f5f9", ec="#64748b"))
        ax.text(xu, 0.7, txt.split(" ")[1], ha="center", va="center", fontsize=9)
        ax.annotate("", (xu, 1.15), (12.15, y0 - 0.05), arrowprops=dict(arrowstyle="-|>", color="#64748b"))
    ax.annotate("", (5.55, y0 - 0.05), (5.55, 1.3), arrowprops=dict(arrowstyle="-|>", color="#64748b", ls="--"))
    ax.text(5.55, 0.95, "Validación LOOCV\n(RMSE, error máx.)", ha="center", fontsize=8.5, color="#334155")
    ax.text(7.5, 4.35, "AirSpline — flujo de datos de la plataforma de sensores virtuales", ha="center",
            fontsize=13, weight="bold", color="#1e3a8a")
    plt.show()

diagrama_bloques()

## 10. KPIs de negocio e impacto
Los parámetros económicos son **supuestos editables** (ajústelos a cotizaciones reales). Referencia sanitaria: guía OMS 2021 para PM₂.₅ en 24 h = **15 µg/m³**.

In [ ]:
# ---- Supuestos económicos (editables) ----
COSTO_ESTACION_USD = 18_000        # costo instalado de una estación de monitoreo PM2.5 (supuesto)
OPEX_ESTACION_USD = 2_500          # mantenimiento/calibración anual por estación (supuesto)
PRECIO_SAAS_MES_USD = 1_200        # suscripción mensual por corredor para municipio/aseguradora (supuesto)
OMS_24H = 15.0                     # µg/m³ guía OMS 2021 (24 h)

n_virtuales = len(xf) - len(x)
capex_evitado = n_virtuales * COSTO_ESTACION_USD
opex_evitado = n_virtuales * OPEX_ESTACION_USD
km_roja = alertas.loc[alertas["Nivel"].str.contains("Roja"), "Longitud (km)"].sum()
km_amar = alertas.loc[alertas["Nivel"].str.contains("Amarilla o"), "Longitud (km)"].sum()

kpis = pd.DataFrame([
    ("Resolución espacial", f"{1/DX:.0f}× (1 km → {DX*1000:.0f} m)", "91 puntos con 10 sensores"),
    ("Sensores virtuales generados", f"{n_virtuales}", "puntos sin hardware"),
    ("CAPEX evitado (supuesto)", f"USD {capex_evitado:,.0f}", f"{n_virtuales} × USD {COSTO_ESTACION_USD:,}"),
    ("OPEX anual evitado (supuesto)", f"USD {opex_evitado:,.0f}/año", "mantenimiento + calibración"),
    ("Continuidad del modelo", "C² (S, S', S'' continuas)", "sin saltos en gradientes"),
    ("RMSE LOOCV (modelo principal)", f"{rmse_best:.2f} µg/m³", MODELO_PRINCIPAL),
    ("Reducción de RMSE vs lineal", f"{REDUCCION_RMSE:.1f} %", "validación leave-one-out"),
    ("Incertidumbre (ancho medio IC95)", f"± {ancho.mean()/2:.2f} µg/m³", f"σ sensor = {SIGMA}"),
    ("Corredor en Alerta ≥ Amarilla", f"{km_amar:.2f} km ({100*km_amar/L:.0f} %)", "exposición moderada-alta"),
    ("Corredor en Alerta Roja", f"{km_roja:.2f} km ({100*km_roja/L:.0f} %)", "suspender actividad al aire libre"),
    ("Concentración media vs OMS", f"{C_media:.1f} µg/m³ = {C_media/OMS_24H:.1f}× guía", "argumento de salud pública"),
    ("Recuperación de inversión cliente", f"{capex_evitado/(12*PRECIO_SAAS_MES_USD):.0f} años de SaaS = CAPEX evitado",
     f"SaaS USD {PRECIO_SAAS_MES_USD:,}/mes"),
], columns=["KPI", "Valor", "Nota"])
display(kpis.style.hide(axis="index").set_properties(**{"text-align": "left"}))

## 11. 🖥️ Panel interactivo AirSpline (demo en vivo para el *Shark Tank*)
Funciona con `ipywidgets` en **VS Code** (extensión Jupyter) y en **Google Colab**. Pestañas:
1. **Mapa de alertas** — cambie método, frontera, umbrales, σ, N y semilla.
2. **Sensor virtual** — consulte cualquier km con su IC95 % y nivel de alerta.
3. **Lecturas en vivo** — edite las 10 estaciones (simule un nuevo día) y recalcule todo.
4. **Validación** — tabla LOOCV recalculada con las lecturas actuales.

In [ ]:
import ipywidgets as W
from IPython.display import display, clear_output, HTML

METODOS_PANEL = ["Spline cúbico", "Hermite (np.gradient)", "PCHIP", "Lineal", "Lagrange (grado n-1)"]

def ajustar(metodo, bc, xs, ys):
    if metodo == "Spline cúbico":
        if bc == "clamped":
            g = np.gradient(ys, xs); return CubicSpline(xs, ys, bc_type=((1, g[0]), (1, g[-1])))
        return CubicSpline(xs, ys, bc_type=bc)
    return construir_modelos(xs, ys)[metodo]

def mc_generico(metodo, bc, xs, ys, sigma, n, semilla):
    rng = np.random.default_rng(semilla)
    curvas = np.array([ajustar(metodo, bc, xs, ys + rng.normal(0, sigma, len(ys)))(xf) for _ in range(n)])
    return curvas

# ---- Controles ----
estilo = {"description_width": "130px"}; lay = W.Layout(width="340px")
w_met = W.Dropdown(options=METODOS_PANEL, value="Spline cúbico", description="Método", style=estilo, layout=lay)
w_bc = W.Dropdown(options=[("Not-a-knot", "not-a-knot"), ("Natural", "natural"), ("Sujeta (np.gradient)", "clamped")],
                  value=BC_PRINCIPAL, description="Frontera spline", style=estilo, layout=lay)
w_ua = W.FloatSlider(value=UMBRAL_AMARILLA, min=30, max=100, step=1, description="Umbral amarilla", style=estilo, layout=lay)
w_ur = W.FloatSlider(value=UMBRAL_ROJA, min=40, max=120, step=1, description="Umbral roja", style=estilo, layout=lay)
w_sig = W.FloatSlider(value=SIGMA, min=0.0, max=10, step=0.5, description="σ ruido (µg/m³)", style=estilo, layout=lay)
w_n = W.IntSlider(value=N_MC, min=20, max=1000, step=20, description="Realizaciones MC", style=estilo, layout=lay)
w_seed = W.IntText(value=SEMILLA, description="Semilla", style=estilo, layout=lay)
w_band = W.Checkbox(value=True, description="Mostrar banda 95 %")
w_km = W.FloatSlider(value=4.0, min=0, max=9, step=0.1, description="Kilómetro", style=estilo,
                     layout=W.Layout(width="600px"), readout_format=".1f")
w_est = [W.BoundedFloatText(value=float(v), min=0, max=500, step=1, description=f"E{i+1} (km {i})",
                            style={"description_width": "90px"}, layout=W.Layout(width="190px")) for i, v in enumerate(y)]
b_reset = W.Button(description="Restaurar datos del caso", icon="undo", button_style="warning")
b_csv = W.Button(description="Exportar malla CSV", icon="download", button_style="info")

out_mapa, out_sensor, out_val, out_msg = W.Output(), W.Output(), W.Output(), W.Output()
estado = {}

def lecturas():
    return np.array([w.value for w in w_est], dtype=float)

def recalcular(*_):
    ys = lecturas(); ua, ur = w_ua.value, w_ur.value
    if ur <= ua: w_ur.value = ua + 1; return
    f = ajustar(w_met.value, w_bc.value, x, ys)
    curvas = mc_generico(w_met.value, w_bc.value, x, ys, w_sig.value, w_n.value, w_seed.value)
    lo, hi = np.percentile(curvas, [2.5, 97.5], axis=0)
    estado.update(f=f, ys=ys, curvas=curvas, lo=lo, hi=hi, ua=ua, ur=ur)
    with out_mapa:
        clear_output(wait=True)
        fig, (ax, axb) = plt.subplots(2, 1, figsize=(12.5, 6), gridspec_kw={"height_ratios": [4, 0.55]}, sharex=True)
        if w_band.value and w_sig.value > 0:
            ax.fill_between(xf, lo, hi, color=C["banda"], alpha=0.5, label="Banda 95 % (Monte Carlo)")
        dibujar_mapa_alertas(ax, f, xf, ua, ur, titulo=f"AirSpline · {w_met.value}" +
                             (f" ({w_bc.value})" if w_met.value == "Spline cúbico" else ""))
        ax.plot(x, ys, "o", color=C["dato"], ms=7, label="Estaciones")
        ax.axvline(w_km.value, color="k", lw=1, ls=":")
        ax.legend(fontsize=8, loc="upper right")
        cf = f(xf); col = np.where(cf >= ur, C["roja"], np.where(cf >= ua, C["amarilla"], "#22c55e"))
        axb.bar(xf, 1, width=DX, color=col); axb.set_yticks([]); axb.grid(False); axb.set_ylabel("Vía", rotation=0, ha="right")
        plt.tight_layout(); plt.show()
        carga = float(f.integrate(0, 9)) if hasattr(f, "integrate") else simpson(cf, x=xf)
        filas = "".join(f"<tr><td>{r['Nivel']}</td><td>{r['Desde km']:.2f}</td><td>{r['Hasta km']:.2f}</td>"
                        f"<td>{r['Longitud (km)']:.2f}</td></tr>" for _, r in tabla_alertas(f, ua, ur).iterrows())
        conf = xf[lo >= ur]; prec = xf[hi >= ur]
        display(HTML(
            f"<div style='font-family:sans-serif'><b>Carga total:</b> {carga:.2f} µg·km/m³ &nbsp;|&nbsp; "
            f"<b>Media del corredor:</b> {carga/L:.2f} µg/m³ &nbsp;|&nbsp; <b>Pico:</b> {cf.max():.1f} µg/m³ en km {xf[cf.argmax()]:.1f}<br>"
            f"<b>Roja confirmada (P2.5≥{ur:.0f}):</b> {'km %.1f–%.1f' % (conf.min(), conf.max()) if len(conf) else '—'} &nbsp;|&nbsp; "
            f"<b>Roja precautoria (P97.5≥{ur:.0f}):</b> {'km %.1f–%.1f' % (prec.min(), prec.max()) if len(prec) else '—'}"
            f"<table style='margin-top:6px;border-collapse:collapse' border='1' cellpadding='4'>"
            f"<tr><th>Nivel</th><th>Desde km</th><th>Hasta km</th><th>Longitud</th></tr>{filas}</table></div>"))
    consultar_km()

def consultar_km(*_):
    if not estado: return
    f, km = estado["f"], w_km.value
    i = int(round(km / DX)); c = float(f(km)); lo, hi = estado["lo"][i], estado["hi"][i]
    p = 100 * (estado["curvas"][:, i] >= estado["ur"]).mean()
    nivel, color = (("ALERTA ROJA", C["roja"]) if c >= estado["ur"] else
                    ("ALERTA AMARILLA", "#eab308") if c >= estado["ua"] else ("NORMAL", "#16a34a"))
    tipo = "Estación física" if abs(km - round(km)) < 1e-9 else "Sensor virtual"
    with out_sensor:
        clear_output(wait=True)
        display(HTML(
            f"<div style='font-family:sans-serif;border:2px solid {color};border-radius:12px;padding:14px;width:560px'>"
            f"<div style='font-size:14px;color:#475569'>{tipo} · km {km:.1f}</div>"
            f"<div style='font-size:40px;font-weight:700;color:{color}'>{c:.1f} µg/m³</div>"
            f"<div style='font-size:18px;font-weight:700;color:{color}'>{nivel}</div>"
            f"<div>IC 95 %: [{lo:.1f}, {hi:.1f}] µg/m³ · P(C ≥ {estado['ur']:.0f}) = {p:.0f} %</div>"
            f"<div>Gradiente local: {float(np.gradient(f(xf), xf)[i]):+.2f} µg/m³/km · {c/OMS_24H:.1f}× guía OMS 24 h</div></div>"))
        fig, ax = plt.subplots(figsize=(6, 2.3))
        ax.hist(estado["curvas"][:, i], bins=20, color=C["banda"], edgecolor="white")
        ax.axvline(c, color=C["spline"], lw=2); ax.axvline(estado["ur"], color=C["roja"], ls="--")
        ax.set(title=f"Distribución Monte Carlo en km {km:.1f}", xlabel="µg/m³"); plt.tight_layout(); plt.show()

def validar(*_):
    with out_val:
        clear_output(wait=True)
        err, res = loocv(x, lecturas())
        display(HTML("<b>Errores absolutos LOOCV (µg/m³)</b>")); display(err.round(3))
        display(HTML("<b>Resumen</b>")); display(res.round(3))

def restaurar(_):
    for w, v in zip(w_est, y): w.value = float(v)

def exportar(_):
    if not estado: return
    f = estado["f"]; cf = f(xf)
    df = pd.DataFrame({"km": xf, "PM25": cf, "IC95_inf": estado["lo"], "IC95_sup": estado["hi"],
                       "Alerta": nivel_alerta(cf, estado["ua"], estado["ur"])})
    df.to_csv("airspline_panel_export.csv", index=False)
    with out_msg:
        clear_output(); print("Guardado: airspline_panel_export.csv")
    if EN_COLAB:
        from google.colab import files; files.download("airspline_panel_export.csv")

for w in [w_met, w_bc, w_ua, w_ur, w_sig, w_n, w_seed, w_band] + w_est:
    w.observe(recalcular, names="value")
for w in w_est: w.observe(validar, names="value")
w_km.observe(consultar_km, names="value")
b_reset.on_click(restaurar); b_csv.on_click(exportar)

encabezado = W.HTML("<div style='background:linear-gradient(90deg,#1e3a8a,#2563eb);color:white;padding:12px 18px;"
                    "border-radius:10px;font-family:sans-serif'><span style='font-size:24px;font-weight:800'>🌬️ AirSpline</span>"
                    "<span style='font-size:14px'> &nbsp;— sensores virtuales de PM₂.₅ con splines cúbicos C² · Air Health Analytics</span></div>")
controles = W.VBox([W.HTML("<b>Modelo</b>"), w_met, w_bc, W.HTML("<b>Política de alertas</b>"), w_ua, w_ur,
                    W.HTML("<b>Incertidumbre (Monte Carlo)</b>"), w_sig, w_n, w_seed, w_band, b_csv, out_msg])
tab_mapa = W.HBox([controles, out_mapa])
tab_sensor = W.VBox([w_km, out_sensor])
tab_datos = W.VBox([W.HTML("Edite las lecturas (µg/m³); todo se recalcula automáticamente."),
                    W.GridBox(w_est, layout=W.Layout(grid_template_columns="repeat(5, 200px)")), b_reset])
tabs = W.Tab(children=[tab_mapa, tab_sensor, tab_datos, out_val])
for i, t in enumerate(["🗺️ Mapa de alertas", "📍 Sensor virtual", "📡 Lecturas en vivo", "✅ Validación LOOCV"]):
    tabs.set_title(i, t)

display(W.VBox([encabezado, tabs]))
recalcular(); validar()

## 12. Resumen ejecutivo y guion del *pitch* (3 minutos)

In [ ]:
r = tramos_sobre_umbral(cs, UMBRAL_ROJA); a = tramos_sobre_umbral(cs, UMBRAL_AMARILLA)
fmt = lambda t: ", ".join(f"km {p:.2f}–{q:.2f}" for p, q in t) or "ninguno"
print(f'''RESUMEN EJECUTIVO — AirSpline (Air Health Analytics)
--------------------------------------------------------------------------------
• Con 10 estaciones físicas se generan {n_virtuales} sensores virtuales (malla de {DX*1000:.0f} m) usando un
  spline cúbico {BC_PRINCIPAL} de clase C², elegido por validación cruzada LOOCV
  (RMSE = {rmse_best:.2f} µg/m³; {REDUCCION_RMSE:.0f} % menor que la interpolación lineal).
• Carga acumulada ∫S dx = {carga_total:.1f} µg·km/m³ → concentración media {C_media:.1f} µg/m³
  ({C_media/OMS_24H:.1f} veces la guía OMS de 24 h). Pico de {float(cs(x_max)):.1f} µg/m³ en el km {x_max:.2f}.
• Alerta Amarilla o superior: {fmt(a)}. Alerta Roja: {fmt(r)}.
• Con ruido instrumental σ = {SIGMA} µg/m³ ({N_MC} simulaciones) la banda 95 % mide ±{ancho.mean()/2:.1f} µg/m³;
  roja confirmada en {'km %.1f–%.1f' % (roja_conf.min(), roja_conf.max()) if len(roja_conf) else '—'} y precautoria en
  {'km %.1f–%.1f' % (roja_prec.min(), roja_prec.max()) if len(roja_prec) else '—'}.
• Infraestructura evitada (supuesto): USD {capex_evitado:,.0f} en CAPEX y USD {opex_evitado:,.0f}/año en OPEX.''')

### Guion del pitch (Ingeniero de Integración y Estrategia)
| Tiempo | Mensaje |
|---|---|
| 0:00–0:30 | **Problema:** 10 sensores cada 1 km dejan *zonas ciegas*; una escuela a 500 m de una estación no sabe si respira aire en alerta roja. |
| 0:30–1:15 | **Solución — AirSpline:** sensores virtuales cada 100 m con splines cúbicos C²: sin oscilaciones de Runge, gradientes continuos, validados con LOOCV. |
| 1:15–2:00 | **Demo en vivo** (panel §11): mover el km, cambiar σ, editar una lectura y ver cómo se mueven las zonas roja/amarilla y su banda de confianza. |
| 2:00–2:30 | **Ventaja competitiva:** alertas con *probabilidad* (Monte Carlo) → decisiones robustas; 81 puntos sin hardware → CAPEX evitado. |
| 2:30–3:00 | **Modelo de negocio y llamado a la acción:** SaaS por corredor para alcaldías, secretarías de salud, aseguradoras y apps de movilidad. *Buscamos piloto en 3 corredores.* |

**Preguntas probables del jurado:** ¿Por qué no Lagrange? (Runge, §2.3/§4) · ¿Por qué not-a-knot y no natural? (LOOCV, §5) · ¿Qué pasa si falla una estación? (LOOCV simula exactamente eso: error máx. reportado) · ¿Qué tan confiable es la alerta? (banda 95 % y P(excedencia), §8) · ¿Limitaciones? (interpolación 1-D, no modela viento/tráfico; extensión: kriging o splines bicúbicos en 2-D).

### Referencias
1. R. L. Burden, J. D. Faires y A. M. Burden, *Análisis Numérico*, 10.ª ed. Cengage Learning, 2017, cap. 3 (interpolación y splines cúbicos).
2. C. de Boor, *A Practical Guide to Splines*, ed. rev. New York: Springer, 2001.
3. P. Virtanen *et al.*, “SciPy 1.0: Fundamental algorithms for scientific computing in Python,” *Nature Methods*, vol. 17, pp. 261–272, 2020.
4. World Health Organization, *WHO Global Air Quality Guidelines: Particulate Matter (PM2.5 and PM10), Ozone, Nitrogen Dioxide, Sulfur Dioxide and Carbon Monoxide*. Geneva: WHO, 2021.
5. F. N. Fritsch y R. E. Carlson, “Monotone piecewise cubic interpolation,” *SIAM J. Numer. Anal.*, vol. 17, no. 2, pp. 238–246, 1980.